# CopyLane 판정 인코더 — 백본 비교 드라이런 (kcbert-base vs KcELECTRA-base)

**목적**: 실제 팀 골든셋(`golden_sample`)이 아직 0건이라 진짜 파인튜닝은 못 한다.
이 노트북은 2차 드라이런(160개 병합 데이터)과 **완전히 같은 데이터·같은 학습 조건**으로
백본만 바꿔서 두 모델을 나란히 돌려본다 — D-70에서 kcbert-base로 확정했지만, D-84에서
KcELECTRA를 비교군으로 남겨둔 것을 실측으로 확인하는 것이 목적이다.

- 비교 백본: `beomi/kcbert-base` (Apache 2.0, D-70 확정 · 주 백본) vs `beomi/KcELECTRA-base` (MIT, D-84 비교군)
- 태스크: 카테고리 판별(4클래스) · **위법 유형(다중 라벨, D-54·D-65)** · 위험도(5값, 순서형) · 질의 의도(5클래스) — 멀티태스크 헤드 (기획서 5-1절)
- 데이터: 직접 작성 60개 + 외부(다른 AI) 생성 CSV 100건 = **160개** (2차 드라이런과 동일, `docs/psj/encoder_dryrun_colab_2차_테스트.ipynb` 참고)
- 근거: `docs/00_설계결정기록.md` D-08·D-65·D-70·D-84·D-131, `app/contracts.py`(Violation enum), `scripts/collect.py`(VIOLATION_TYPES/CANDIDATE_TYPES), `docs/01_기획/02_프로젝트기획서.md` 5-1·5-5절

⚠️ 여기서 나오는 정확도·loss 숫자는 **의미 없다** — 표본이 수십~백여 건뿐이라 파이프라인·상대
비교 검증용이다. 절대 성능 수치로 인용하지 않는다. 두 모델을 같은 SEED·같은 데이터·같은
epoch 수로 돌려서 **상대적인 차이**(과적합 양상, dev 매치율)만 참고한다.

## 0. 환경 설정 — 코랩에서 실행

In [ ]:
!pip install -q transformers torch scikit-learn matplotlib

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import random

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

## 1. 위법 유형 라벨 체계 — `app/contracts.py`의 `Violation` 그대로

기존 노트북은 위법 유형을 반영하지 않고 `risk`(위험도 5값)만 뒀었다. 실제 스키마는
`app/contracts.py`의 `Violation` enum(팀장 소유, `db/schema.sql`의 `violation_t`와 동일해야 함 — D-54)
이 정답이고, **다중 라벨**이다(한 문장에 여러 유형이 동시에 걸릴 수 있음).

🚨 **11종 전부가 지금 학습 대상은 아니다.** `scripts/collect.py`의 주석(D-65):

> 뒤 다섯은 **편입 후보**다 — 인코더가 예측하는 **확정 클래스는 `VIOLATION_TYPES` 6종**이고,
> **승격 판정일은 2026-09-17**(오늘)이다.

판정 규칙은 D-40·D-65 기준 유형별 **30건 미만 = 측정 불가**다. 첨부된 실측 차트
(`test_sentence` 206건, 유형별 평가 가능 하한 30건)가 이 판정의 근거 데이터로 보인다 —
`거짓_과장`(73)·`소비자_기만`(67)만 30건을 넘고, 나머지는 전부 하한 미달(부당_비교광고 4·비방광고 2·나머지 0).

In [ ]:
# app/contracts.py 의 Violation enum 그대로 — db/schema.sql violation_t 와 동일해야 함 (D-54)

# D-65 확정 6종 — scripts/collect.py VIOLATION_TYPES. 인코더가 실제로 예측하는 클래스.
VIOLATION_TYPES = [
    "질병_예방치료_표방",
    "건강기능식품_오인",
    "의약품_오인",
    "거짓_과장",
    "소비자_기만",
    "후기_체험기_기만",
]

# 편입 후보 5종 — scripts/collect.py CANDIDATE_TYPES. 아직 정식 학습 클래스가 아니다.
# (2026-09-17 실측: 거짓_과장 73 · 소비자_기만 67 만 30건 하한 통과, 나머지는 하한 미달)
CANDIDATE_TYPES = [
    "추천_보증_뒷광고",
    "부당_비교광고",      # 실측 4건 — 하한 미달
    "비방광고",           # 실측 2건 — 하한 미달
    "실증책임_위반",
    "기능성화장품_오인",
]

ALL_VIOLATION_TYPES = VIOLATION_TYPES + CANDIDATE_TYPES
MIN_SAMPLES = 30  # D-40 — 이 미만은 '측정 불가'로 취급, Accuracy로 읽지 않는다

print(f"확정 클래스(학습 대상): {len(VIOLATION_TYPES)}종")
print(f"편입 후보(참고용, 이번 드라이런 제외): {len(CANDIDATE_TYPES)}종")

⚠️ **이번 드라이런은 `VIOLATION_TYPES` 확정 6종만 학습한다.** 편입 후보 5종은 D-65 판정
(오늘) 이후 팀 결정에 따라 편입되거나 「측정 불가 유형」으로 남는다 — 지금 섞어서 학습하면
나중에 클래스 집합이 바뀔 때 다시 갈아엎어야 한다(D-08 「학습 중 클래스가 바뀌면 처음부터
다시」).

## 2. 직접 작성 예시 데이터 (60개)

실제 결함 주입 골든셋(D-25·D-74, TRANSFORM_RULES T1~T10)을 흉내 낸 **직접 작성한 샘플**이다.
실제 데이터가 아니므로 라벨 분포·표현이 거칠다 — 파이프라인 검증용으로만 쓴다.

**1차 드라이런(20개) 대비 바뀐 점** — 지난 실행에서 위법 유형 헤드가 클래스 불균형
(위반없음 12/20 = 60%) 때문에 3문장 전부 `(없음)`으로만 예측하는 문제가 있었다. 이번엔:

- 20 → **60개**로 표본 확대
- 위반없음 비율을 60% → 약 30%로 낮춤 — 나머지는 6개 확정 유형에 최소 6건씩 고르게 배분
- 유형별 단독 사례뿐 아니라 **다중 라벨 조합**(2~3개 유형 동시)도 늘림 — 실제 위반 문구는 여러 유형이 겹치는 경우가 흔함(D-74 변환 규칙표 참고)

(이 60개는 직접 작성분이고, 아래 2-1절에서 외부 CSV 100건을 추가로 병합해 총 160개가 된다.
train/dev 분리는 2-2절에서 병합 후 한 번에 수행한다.)

라벨 스키마 (기획서 5-1절 요약):
- `category`: 0=일반, 1=식품, 2=건기식, 3=화장품
- `violations`: `VIOLATION_TYPES` 6종에 대한 **다중 라벨**(멀티핫) — 위 1장 참고
- `risk`: 0=특이사항없음 1=주의 2=업무정지위험 3=과징금위험 4=형사위험 (순서형, `violations`와 별개 축)
- `intent`: 0=문구검수 1=법령조회 2=사례검색 3=대체문구요청 4=복합

In [ ]:
# (text, category, violations(멀티핫 6자리), risk, intent)
# violations 순서 = VIOLATION_TYPES 순서: [질병_예방치료_표방, 건강기능식품_오인, 의약품_오인, 거짓_과장, 소비자_기만, 후기_체험기_기만]
DUMMY_SAMPLES = [
    # --- 위반없음 (18건 ≈ 30%) ---
    ("이 크림은 피부 진정에 도움을 줄 수 있다고 알려져 있습니다", 3, [0,0,0,0,0,0], 0, 0),
    ("제26조가 어떤 내용인가요", 0, [0,0,0,0,0,0], 0, 1),
    ("비슷한 위반 사례가 있을까요", 0, [0,0,0,0,0,0], 0, 2),
    ("이 문구 대신 쓸 표현을 추천해주세요", 0, [0,0,0,0,0,0], 0, 3),
    ("이거 왜 위반인지랑 고친 문구도 같이 알려주세요", 0, [0,0,0,0,0,0], 0, 4),
    ("임상시험으로 효과가 입증된 성분 함유", 2, [0,0,0,0,0,0], 1, 0),          # 경계 클래스(실증 있음)
    ("성분표만 보면 이 제품이 합법인지 알 수 있나요", 0, [0,0,0,0,0,0], 0, 1),
    ("이런 표현으로 바꾸면 안전한가요 검토도 부탁해요", 0, [0,0,0,0,0,0], 0, 4),
    ("영양성분 기준에 맞게 표시했습니다", 1, [0,0,0,0,0,0], 0, 0),
    ("기능성 원료 함유로 도움을 줄 수 있습니다", 2, [0,0,0,0,0,0], 0, 0),      # 인정 기능성 문구 그대로
    ("과징금 처분 기준이 궁금합니다", 0, [0,0,0,0,0,0], 0, 1),
    ("작년에 비슷한 문구로 처분받은 사례 있나요", 0, [0,0,0,0,0,0], 0, 2),
    ("보존료 함량을 제품 라벨에 기재했습니다", 1, [0,0,0,0,0,0], 0, 0),
    ("자외선 차단 효과가 있는 제품입니다 (SPF30 실증 완료)", 3, [0,0,0,0,0,0], 0, 0),
    ("이 제품의 알레르기 유발 성분을 알려주세요", 0, [0,0,0,0,0,0], 0, 1),
    ("경쟁 제품 사례도 같이 검토해줄 수 있나요", 0, [0,0,0,0,0,0], 0, 2),
    ("이 문구는 표시광고법상 문제 없나요", 0, [0,0,0,0,0,0], 0, 0),
    ("할인 행사 안내 문구인데 검토 부탁드립니다", 1, [0,0,0,0,0,0], 0, 0),

    # --- 질병_예방치료_표방 단독/조합 (8건) ---
    ("이 크림 바르면 아토피가 완치됩니다", 3, [1,0,0,0,0,0], 4, 0),
    ("매일 드시면 당뇨가 낫습니다", 1, [1,0,0,0,0,0], 4, 0),
    ("이 성분이 암 예방에 직접적 효과가 있습니다", 2, [1,0,0,1,0,0], 4, 0),   # +거짓_과장
    ("고혈압 치료에 탁월한 건강기능식품입니다", 2, [1,1,0,0,0,0], 4, 0),      # +건강기능식품_오인
    ("이 로션으로 아토피 피부염을 근본 치료하세요", 3, [1,0,0,0,0,0], 4, 0),
    ("관절염 통증이 완전히 사라지는 영양제", 2, [1,0,0,1,0,0], 4, 0),        # +거짓_과장
    ("불면증을 치료해주는 수면 유도 성분", 1, [1,0,0,0,0,0], 4, 0),
    ("비염 예방과 치료에 효과적인 스프레이", 1, [1,0,0,0,0,0], 4, 0),

    # --- 건강기능식품_오인 단독/조합 (7건) ---
    ("이 영양제 드시면 면역력이 확실히 증진됩니다", 2, [0,1,0,1,0,0], 2, 0),  # +거짓_과장(T1)
    ("이 제품은 건강기능식품 인증 성분과 동일한 효과", 1, [0,1,0,0,0,0], 2, 0),
    ("일반식품이지만 건기식만큼 면역에 좋습니다", 1, [0,1,0,0,0,0], 2, 0),
    ("이 음료는 기능성 원료 함량이 건기식 기준을 충족합니다", 1, [0,1,0,0,0,0], 1, 0),
    ("건강기능식품 수준의 항산화 효과를 가진 일반식품", 1, [0,1,0,1,0,0], 2, 0),  # +거짓_과장
    ("이 제품 섭취만으로 건기식 인정 기능성을 대체합니다", 1, [0,1,0,0,0,0], 2, 0),
    ("체지방 감소 효과가 건강기능식품과 동등합니다", 1, [0,1,0,1,0,0], 2, 0),  # +거짓_과장

    # --- 의약품_오인 단독/조합 (7건) ---
    ("이 제품은 국내 유일 특허 치료제입니다", 1, [0,0,1,1,0,0], 4, 0),        # +거짓_과장(T4)
    ("처방전 없이도 살 수 있는 의약품급 효능", 2, [0,0,1,0,0,0], 4, 0),
    ("이 연고는 상처 치료제와 동일한 성분입니다", 3, [0,0,1,0,0,0], 4, 0),
    ("항염 치료 효과가 있는 특효 성분 함유", 2, [0,0,1,1,0,0], 4, 0),        # +거짓_과장
    ("약국에서 파는 소화제만큼 효과가 확실합니다", 1, [0,0,1,0,0,0], 3, 0),
    ("이 크림은 피부과 처방약 수준의 치료 효과", 3, [0,0,1,0,0,0], 4, 0),
    ("감기약 대신 먹어도 되는 즉효 성분", 1, [0,0,1,0,0,0], 4, 0),

    # --- 거짓_과장 단독/조합 (8건) ---
    ("다들 드시고 확실히 효과 보셨어요", 1, [0,0,0,1,1,0], 1, 0),            # +소비자_기만(T9)
    ("경쟁 제품보다 훨씬 효과가 좋습니다", 1, [0,0,0,1,0,0], 2, 0),
    ("세포 재생에 탁월한 효과가 있는 안티에이징 크림", 3, [0,0,0,1,0,0], 2, 0),
    ("먹기만 하면 살이 쫙 빠집니다", 1, [0,0,0,1,1,0], 4, 0),                # +소비자_기만
    ("100% 완치 보장, 유일무이한 효능", 2, [0,0,0,1,0,0], 3, 0),
    ("업계 최고 효과, 타사 제품은 비교 불가", 1, [0,0,0,1,0,0], 2, 0),
    ("놀라운 즉각 효과, 사용 즉시 체감", 3, [0,0,0,1,0,0], 1, 0),
    ("과학적으로 100% 입증된 최고의 성분", 2, [0,0,0,1,0,0], 2, 0),

    # --- 소비자_기만 단독/조합 (6건) ---
    ("이 제품 쓰고 인생이 바뀌었어요", 3, [0,0,0,0,1,0], 1, 0),
    ("고객님들이 재구매율 100%라고 극찬한 제품", 1, [0,0,0,1,1,0], 2, 0),    # +거짓_과장
    ("원가 그대로 드리는 마지막 특가입니다", 1, [0,0,0,0,1,0], 1, 0),
    ("실제로 다 나았다는 후기가 압도적입니다", 2, [0,0,0,0,1,0], 1, 0),
    ("전문가도 인정한 효과, 모두가 만족했습니다", 1, [0,0,0,1,1,0], 2, 0),   # +거짓_과장
    ("이 결과는 개인차 없이 누구나 동일합니다", 2, [0,0,0,0,1,0], 1, 0),

    # --- 후기_체험기_기만 단독/조합 (6건) ---
    ("체험단 후기입니다 정말 만족스러워요 (광고 표시 없음)", 1, [0,0,0,0,0,1], 1, 0),
    ("실사용 후기 이렇게 좋을 줄 몰랐어요 (협찬 미표시)", 3, [0,0,0,0,0,1], 1, 0),
    ("내돈내산이라고 적었지만 사실 협찬받았습니다", 1, [0,0,0,0,1,1], 1, 0), # +소비자_기만
    ("블로거 체험단 모집 후기, 대가 지급 사실 비공개", 2, [0,0,0,0,0,1], 1, 0),
    ("인플루언서 추천이라고만 표시하고 광고임을 숨김", 1, [0,0,0,0,0,1], 1, 0),
    ("체험 후기 게시물인데 실제로는 유료 광고입니다", 3, [0,0,0,0,0,1], 1, 0),
]

print(f"직접 작성 샘플 수: {len(DUMMY_SAMPLES)}")

## 2-1. 다른 AI가 만든 데이터 CSV 병합

팀 골든셋(D-25 결함 주입) 대신, 다른 AI에게 같은 라벨 스키마로 만들어달라고 요청한
CSV를 합쳐서 표본을 더 늘린다. CSV 컬럼: `text, category, violations, risk, intent`
(`violations`는 `\"1,0,0,0,0,0\"` 형태의 6자리 쉼표 문자열).

**주의** — 이것도 결국 AI가 지어낸 문장이라 D-25가 말하는 「고시의 금지 유형에서
도출한 정답」은 아니다. 다만 문장 표현이 저와 다른 AI 두 소스로 섞이면서 다양성은
늘어난다 — 여전히 파이프라인·클래스 불균형 검증용이지 성능 검증용은 아니다.

In [ ]:
from google.colab import files
print("CSV 파일을 선택하세요 (예: 광고문구_위법판정_드라이런_100건.csv)")
uploaded = files.upload()

In [ ]:
import csv, io

def load_csv_samples(filename):
    samples = []
    with open(filename, encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            text = row["text"].strip()
            if not text:
                continue
            category = int(row["category"])
            violations = [int(x) for x in row["violations"].split(",")]
            risk = int(row["risk"])
            intent = int(row["intent"])
            assert len(violations) == 6, f"violations 길이 오류: {row}"
            samples.append((text, category, violations, risk, intent))
    return samples

csv_filename = list(uploaded.keys())[0]
NEW_SAMPLES = load_csv_samples(csv_filename)
print(f"CSV에서 불러온 샘플: {len(NEW_SAMPLES)}개")

DUMMY_SAMPLES = DUMMY_SAMPLES + NEW_SAMPLES
print(f"합친 후 총 샘플: {len(DUMMY_SAMPLES)}개")

# 클래스 분포 재확인 — 합친 뒤에도 불균형이 심하지 않은지
n_no_violation = sum(1 for s in DUMMY_SAMPLES if sum(s[2]) == 0)
print(f"\n위반없음: {n_no_violation}건 ({n_no_violation/len(DUMMY_SAMPLES)*100:.0f}%)")
for i, vtype in enumerate(VIOLATION_TYPES):
    count = sum(1 for s in DUMMY_SAMPLES if s[2][i] == 1)
    print(f"  {vtype}: {count}건")

## 2-2. train/dev 분리 (셔플 후 분리)

CSV를 뒤에 이어붙였기 때문에 그냥 마지막 N건을 자르면 dev가 CSV 쪽 문장으로만
채워져 편향된다. 이번엔 **셔플 후 80/20 분리**로 바꾼다 (SEED 고정으로 재현 가능).

In [ ]:
import random as _random

indices = list(range(len(DUMMY_SAMPLES)))
_random.Random(SEED).shuffle(indices)

N_DEV = max(1, round(len(DUMMY_SAMPLES) * 0.2))
dev_idx = set(indices[:N_DEV])
train_idx = [i for i in indices if i not in dev_idx]
dev_idx = [i for i in indices if i in dev_idx]

def gather(idx_list):
    texts_ = [DUMMY_SAMPLES[i][0] for i in idx_list]
    cat_ = [DUMMY_SAMPLES[i][1] for i in idx_list]
    viol_ = [DUMMY_SAMPLES[i][2] for i in idx_list]
    risk_ = [DUMMY_SAMPLES[i][3] for i in idx_list]
    intent_ = [DUMMY_SAMPLES[i][4] for i in idx_list]
    return texts_, cat_, viol_, risk_, intent_

train_texts, train_cat, train_violation, train_risk, train_intent = gather(train_idx)
dev_texts, dev_cat, dev_violation, dev_risk, dev_intent = gather(dev_idx)

print(f"train: {len(train_texts)}건 · dev: {len(dev_texts)}건")

# dev 세트도 위반없음 비율이 너무 쏠리지 않았는지 확인
n_dev_no_violation = sum(1 for v in dev_violation if sum(v) == 0)
print(f"dev 중 위반없음: {n_dev_no_violation}/{len(dev_violation)}")

## 3. 백본 두 개 후보 정의 — kcbert-base vs KcELECTRA-base

D-70에서 kcbert-base를 주 백본으로 확정했지만, D-84에서 같은 저자·같은 도메인(댓글체)·
비슷한 크기(0.1B급)인 KcELECTRA-base를 비교군으로 남겨뒀다. 두 모델을 **완전히 같은
데이터·같은 SEED·같은 epoch 수**로 순서대로 돌려서 loss·dev 매치율을 나란히 비교한다.

In [ ]:
BACKBONE_CANDIDATES = {
    "kcbert-base": "beomi/kcbert-base",        # D-70 확정 — 주 백본
    "KcELECTRA-base": "beomi/KcELECTRA-base",  # D-84 비교군 — MIT, 같은 저자·도메인
}
print(BACKBONE_CANDIDATES)

## 4. 멀티태스크 헤드 정의 (백본 비의존)

기획서 5-1절 태스크 표를 그대로 반영 — 카테고리(4클래스 분류) · **위법 유형(6클래스 다중 라벨)**
· 위험도(5차원 순서형) · 의도(5클래스 분류)를 백본 인코더 위에 헤드 4개로 얹는다.
`JudgeEncoder`는 어떤 BERT 계열 백본이든(kcbert·KcELECTRA 둘 다 `AutoModel`로 로드되는
표준 인코더 구조라) 그대로 재사용할 수 있게 설계한다.

위법 유형은 **다중 라벨**이라 softmax+CrossEntropy가 아니라 **sigmoid+BCE**로 학습한다.

실제 근거 스팬 헤드(BIO, D-131)는 이번 드라이런에서는 생략한다 — 문장 단위 분류/회귀
파이프라인이 두 백본에서 동일하게 도는지만 먼저 확인한다.

In [ ]:
class JudgeEncoder(nn.Module):
    """기획서 5-1절 멀티태스크 헤드 — 카테고리 / 위법유형(다중라벨) / 위험도(순서형) / 의도."""
    def __init__(self, backbone, n_category=4, n_violation=6, n_risk=5, n_intent=5):
        super().__init__()
        self.backbone = backbone
        hidden = backbone.config.hidden_size
        self.category_head = nn.Linear(hidden, n_category)
        self.violation_head = nn.Linear(hidden, n_violation)  # 다중 라벨 — sigmoid+BCE
        self.risk_head = nn.Linear(hidden, n_risk)             # 순서형이지만 드라이런에서는 분류로 단순화
        self.intent_head = nn.Linear(hidden, n_intent)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]  # [CLS]
        return {
            "category": self.category_head(cls),
            "violation": self.violation_head(cls),  # 로짓 — 학습 loss에서 sigmoid 적용
            "risk": self.risk_head(cls),
            "intent": self.intent_head(cls),
        }

## 5. 데이터셋 정의 (백본 비의존 — 토크나이저만 모델별로 교체)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class DummyJudgeDataset(Dataset):
    def __init__(self, texts, cat, violations, risk, intent, tokenizer, max_len=64):
        self.texts, self.cat, self.violations = texts, cat, violations
        self.risk, self.intent = risk, intent
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_len, return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "category": torch.tensor(self.cat[idx], dtype=torch.long),
            "violation": torch.tensor(self.violations[idx], dtype=torch.float),  # 멀티핫 — BCE용 float
            "risk": torch.tensor(self.risk[idx], dtype=torch.long),
            "intent": torch.tensor(self.intent[idx], dtype=torch.long),
        }

## 6. 모델별 학습 + 평가 함수 — 두 백본에 동일하게 적용

1차(60개)는 과적합(dev loss 발산), 2차(160개)는 완화를 확인했다. 이번엔 2차와 동일한
**160개·20epoch·셔플 80/20 분리** 조건에서 백본만 바꿔가며 같은 함수로 돌린다 —
조건을 고정해야 백본 차이만 비교할 수 있다.

위법 유형만 손실 함수가 다르다(BCEWithLogitsLoss — 다중 라벨) — 나머지 세 헤드는 CrossEntropyLoss.

In [ ]:
EPOCHS = 20  # 드라이런용 — 2차와 동일 조건

def run_epoch(model, loader, optimizer, ce_loss_fn, bce_loss_fn, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            cat = batch["category"].to(device)
            violation = batch["violation"].to(device)
            risk = batch["risk"].to(device)
            intent = batch["intent"].to(device)

            if train:
                optimizer.zero_grad()
            out = model(input_ids, attention_mask)
            loss = (
                ce_loss_fn(out["category"], cat)
                + bce_loss_fn(out["violation"], violation)
                + ce_loss_fn(out["risk"], risk)
                + ce_loss_fn(out["intent"], intent)
            )
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
    return total_loss / len(loader)


def predict(model, tokenizer, sent):
    enc = tokenizer(sent, truncation=True, padding="max_length", max_length=64, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(enc["input_ids"], enc["attention_mask"])
    cat_pred = out["category"].argmax(-1).item()
    risk_pred = out["risk"].argmax(-1).item()
    intent_pred = out["intent"].argmax(-1).item()
    violation_probs = torch.sigmoid(out["violation"]).squeeze(0).tolist()
    violation_preds = [VIOLATION_TYPES[i] for i, p in enumerate(violation_probs) if p >= 0.5]
    return cat_pred, violation_preds, risk_pred, intent_pred


def run_backbone_trial(model_name, label):
    """백본 하나를 로드→학습→평가까지 전부 수행하고 결과 dict를 반환한다."""
    print(f"\n{'='*60}\n[{label}] {model_name} 로드 중...\n{'='*60}")
    torch.manual_seed(SEED)  # 두 모델의 초기화·셔플 조건을 최대한 맞춘다

    tok = AutoTokenizer.from_pretrained(model_name)
    backbone = AutoModel.from_pretrained(model_name)
    model = JudgeEncoder(backbone).to(device)

    train_dataset = DummyJudgeDataset(train_texts, train_cat, train_violation, train_risk, train_intent, tok)
    dev_dataset = DummyJudgeDataset(dev_texts, dev_cat, dev_violation, dev_risk, dev_intent, tok)
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    dev_loader = DataLoader(dev_dataset, batch_size=8, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    ce_loss_fn = nn.CrossEntropyLoss()
    bce_loss_fn = nn.BCEWithLogitsLoss()

    history_train, history_dev = [], []
    for epoch in range(EPOCHS):
        train_loss = run_epoch(model, train_loader, optimizer, ce_loss_fn, bce_loss_fn, train=True)
        history_train.append(train_loss)
        if (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:
            dev_loss = run_epoch(model, dev_loader, optimizer, ce_loss_fn, bce_loss_fn, train=False)
            history_dev.append((epoch + 1, dev_loss))
            print(f"[{label}] epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  dev_loss={dev_loss:.4f}")
        else:
            print(f"[{label}] epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}")

    # dev 세트 정답 매치율
    model.eval()
    n_match = 0
    for i, sent in enumerate(dev_texts):
        cat_pred, violation_preds, risk_pred, intent_pred = predict(model, tok, sent)
        gold_violations = [VIOLATION_TYPES[j] for j, v in enumerate(dev_violation[i]) if v == 1]
        if set(violation_preds) == set(gold_violations):
            n_match += 1
    dev_match_rate = n_match / len(dev_texts)
    print(f"[{label}] dev 위법유형 매치율: {n_match}/{len(dev_texts)} ({dev_match_rate*100:.1f}%)")

    # 체크포인트 저장
    ckpt_path = f"/content/checkpoints/judge_encoder_{label}.pt"
    os.makedirs("/content/checkpoints", exist_ok=True)
    torch.save(model.state_dict(), ckpt_path)

    return {
        "label": label,
        "model_name": model_name,
        "model": model,
        "tokenizer": tok,
        "history_train": history_train,
        "history_dev": history_dev,
        "dev_match_rate": dev_match_rate,
        "ckpt_path": ckpt_path,
    }

## 7. 두 백본 순차 학습 실행

코랩 GPU 메모리 절약을 위해 한 번에 하나씩 로드·학습·평가하고 다음 모델로 넘어간다
(동시에 두 모델을 GPU에 올리지 않는다).

In [ ]:
import os

results = {}
for label, model_name in BACKBONE_CANDIDATES.items():
    results[label] = run_backbone_trial(model_name, label)
    # GPU 메모리 정리 — 다음 백본 로드 전
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 8. 비교 결과 — loss 추이 · dev 매치율

같은 데이터·같은 SEED·같은 epoch 수 조건에서 두 백본의 학습 곡선과 dev 세트 매치율을
나란히 정리한다. 이 숫자 자체는 표본이 적어 절대 성능으로 볼 수 없지만, **어느 쪽이
이 소량 데이터에서 더 잘 수렴하는지의 상대적 경향**은 참고할 수 있다.

In [ ]:
print(f"{'백본':<16}{'최종 train loss':<18}{'최종 dev loss':<16}{'dev 매치율':<12}")
print("-" * 62)
for label, r in results.items():
    final_train = r["history_train"][-1]
    final_dev = r["history_dev"][-1][1] if r["history_dev"] else float("nan")
    print(f"{label:<16}{final_train:<18.4f}{final_dev:<16.4f}{r['dev_match_rate']*100:<10.1f}%")

print("\n=== epoch별 dev loss 추이 (5epoch 간격) ===")
for label, r in results.items():
    print(f"{label}: {[(ep, round(l, 3)) for ep, l in r['history_dev']]}")

In [ ]:
# 그래프로도 확인 — train/dev loss 곡선을 백본별로 겹쳐 그린다
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for label, r in results.items():
    ax[0].plot(range(1, EPOCHS + 1), r["history_train"], label=label)
    dev_epochs = [ep for ep, _ in r["history_dev"]]
    dev_losses = [l for _, l in r["history_dev"]]
    ax[1].plot(dev_epochs, dev_losses, marker="o", label=label)

ax[0].set_title("train loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].set_title("dev loss (5epoch 간격)"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout()
plt.show()

## 9. 정리 — 이 비교 드라이런이 확인한 것 / 안 한 것

**확인한 것 (파이프라인·상대 비교)**
- 두 백본(kcbert-base·KcELECTRA-base) 모두 같은 `JudgeEncoder` 헤드 구조에서 forward pass·학습 루프 정상 동작
- 동일 데이터(160개)·동일 SEED·동일 epoch(20) 조건에서 두 백본의 train/dev loss 곡선과 dev 위법유형 매치율을 나란히 확보
- 체크포인트 저장 경로도 백본별로 분리 가능함을 확인 (`judge_encoder_kcbert-base.pt` 등)

**확인하지 않은 것 (실제 채택 판단 관련 — 팀 골든셋 공급 후 필요)**
- 진짜 일반화 성능 비교 — dev 32건은 참고용이지 정식 test_holdout이 아니다(D-25)
- 두 백본의 응답시간(p95) 실측 비교 — 기획서 7-1절 예산(카테고리 <100ms·위법유형 <150ms) 대비 실측 필요
- DAPT(도메인 적응 사전학습) 적용 시 두 백본의 격차가 좁혀지는지 여부
- Span F1·MAE·QWK·selective risk 등 D-77 공식 평가지표 기준 비교
- 라이선스는 이미 확정 조건 — kcbert Apache 2.0, KcELECTRA MIT 모두 배포 가능(D-70·D-84)이므로 이번 비교는 순수 성능·수렴 속도만 본다

### 다음 단계
1. 이 노트북 결과(어느 백본이 소량 데이터에서 더 안정적으로 수렴했는지)를 팀에 공유
2. 실제 골든셋 공급 시 같은 비교를 동일 구조로 재실행 — 결과가 뒤집히는지 확인
3. 우세한 쪽이 있어도 D-70 결정을 뒤집는 것은 팀 논의 필요 — 이 노트북은 판단 근거 자료일 뿐, 자동 채택 기준이 아님